<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2023notebooks/2020_0720tlpa_sala_resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CCAP Project TLPA and SALA combined

- title: 転移学習による TLPA 画像認識
- author: 浅川伸一
- filename: `2020-0712tlpa_resnet18.ipynb`
- last date: 2020-0713
- note:
    - 使用モデル: ResNet-18, 論文: https://arxiv.org/abs/1512.03385
    - データ: TLPA 図版 大門正太郎 先生より
    
## 手順

1. データを定義する。PyTorch の場合 dataloader を使うために DataSet の定義を必要とする
2. モデルを定義する。転移学習の場合，訓練済モデルが公開されているので，そのモデルを使う。
    どのモデルを使ってもよいのだが，`model = models.vgg16(weights="DEFAULT", progress=True)` のようなモデルを使う
3. 最終層を，解くべき問題のクラス数に合わせて付け替える
4. 損失関数を定義する
5. 最適化関数を定義する
6. 訓練を実行する
7. 評価する

という流れになる

In [1]:
import torch
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

#import sys
from termcolor import colored
#  print(colored('Miss', 'red'), end="")

In [ ]:
# 各画像の画面表示時に日本語キャプションを付与する準備
import matplotlib.pyplot as plt
%matplotlib inline
!pip install japanize-matplotlib
import japanize_matplotlib

#  ImageNet の各ラベルの WordNet ID 処理用
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

#ライブラリのインストール
!git clone https://github.com/project-ccap/ccap.git

#付属のデータをダウンロードする
!wget --no-check-certificate --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1xKXbovkEQwdJefzCuaS_a351LUIuRz-1' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1xKXbovkEQwdJefzCuaS_a351LUIuRz-1" -O ccap_data.tgz && rm -rf /tmp/cookies.txt
!tar xzf ccap_data.tgz


## 1. データの定義

全データ all_data にたいして以下の辞書を定義:

- num2id: 全データ番号 (num) と ラベル名の数字である ID 番号との関係
- num2owrd: 全データ番号 (num) からラベル名 word を検索する辞書
- id2word: ID 番号から ラベル名 (目標語 word) から検索する辞書
- word2id: ラベル名 (目標語 word) から付番した ID 番号を検索する辞書

id 番号と num 番号ともに数字であるが， id < num である。
num は全データに対してユニークに付番されている。
一方 id は label に対応している。
すなわち総単語数，あるいは総語彙数である。

In [3]:
def combine_tlpa_sala():
    from ccap import tlpaDataset
    from ccap import salaDataset

    tlpa = tlpaDataset()
    sala = salaDataset()

    #print(sala.data[34])
    #print(sala.labels[34])
    sala.data[34]['label'] = 'すりこ木'
    sala.labels[34] = 'すりこ木'

    tlpa_set = set(tlpa.labels)
    sala_set = set(sala.labels)

    # TLPA の単語から連番を得るための辞書
    tlpa_word2num = {tlpa.data[x]['label']:x for x in tlpa.data}

    # SALA の単語から連番を得るための辞書
    sala_word2num = {sala.data[x]['label']:x for x in sala.data}

    # TLPA と SALA  で重複する単語を探す
    dup_words = []
    counter = 0
    for i, word in enumerate(tlpa.labels):
        if word in sala_set:
            counter += 1
            dup_words.append(word)

    print('There are ', colored(len(dup_words), 'red'), 'duplicate images.')

    # TLPA と SALA とで共通するデータベース all_data の作成
    num = 0
    img_file_list = []
    ## 最初は TLPA
    num2word = {}
    for word in tlpa.labels:
        num2word[num] = word
        img_file_list.append(tlpa.data[num]['img'])
        num +=1

    ##  続いて SALA ,  num はリセットしない
    for j, word in enumerate(sala.labels):
        num2word[num] = word
        img_file_list.append(sala(j)[0])
        num += 1

    word2id, _id = {}, 0
    for num in num2word:
        word = num2word[num]
        if not word in word2id:
            word2id[word] = _id
            _id += 1

    id2word = {_id:word for word, _id in word2id.items()}
    num2id = {num:word2id[num2word[num]] for num in num2word}

    print(dup_words)
    print('TLPA words:', tlpa.__len__())
    print('SALA words:', sala.__len__())
    print('Duplicated words:', len(dup_words))
    print('Number of data :', tlpa.__len__() + sala.__len__())
    out_features = tlpa.__len__() + sala.__len__() - len(dup_words)
    print('Number of labels:', out_features)
    print('## 上の数が出力層のニューロン数になる')

    #print(img_file_list)
    #print(num2id)
    #print(out_features)
    return img_file_list, num2id, num2word, id2word, out_features, dup_words

In [ ]:
import IPython
isColab = 'colab' in str(IPython.get_ipython())
if isColab:
    !pip install gensim

In [ ]:
_img_file_list, _num2id, _num2word, _id2word, _out_features, _dup_words = combine_tlpa_sala()
#print(_img_list)
#print(_num2id)
#print(_dup_words)
#print(_out_features)

In [ ]:
# # from https://github.com/dmlc/xgboost/issues/1715
# # ただし，https://software.intel.com/en-us/comment/1579647 によれば，Intel CPU の問題
# import os
# import platform

# if platform.system() == 'Darwin':
#     if os.environ.__contains__('KMP_DUPLICATE_LIB_OK') == False or os.environ['KMP_DUPLICATE_LIB_OK'] == 'False':
#         os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
%matplotlib inline
import sys
import numpy as np
import json
import random
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import japanize_matplotlib

import skimage.color  # for gray2rgb()
from scipy.special import logsumexp, softmax
import importlib  # for debugging
from termcolor import colored

#import torch
import torchvision
from torchvision import models, transforms
import torch.utils.data as data
import torch.nn as nn
import torch.optim as optim

# PyTorch バージョン確認
print("PyTorch Version: ",torch.__version__)
print("Torchvision Version: ",torchvision.__version__)

## 2.  モデルの定義

In [ ]:
# 各モデルを定義し，訓練済み結合係数をダウンロードする
DNNs = {}
DNNs['resnet18'] = models.resnet18(weights="DEFAULT", progress=True)
# DNNs['alexnet'] = models.alexnet(weights="DEFAULT", progress=True)
# DNNs['vgg16'] = models.vgg16(weights="DEFAULT", progress=True)
# DNNs['squeezenet']= models.squeezenet1_0(weights="DEFAULT", progress=True)
# DNNs['densenet'] = models.densenet161(weights="DEFAULT", progress=True)
# DNNs['inception'] = models.inception_v3(weights="DEFAULT", progress=True)
# DNNs['googlenet'] = models.googlenet(weights="DEFAULT", progress=True)
# DNNs['shufflenet'] = models.shufflenet_v2_x1_0(weights="DEFAULT", progress=True)
# DNNs['mobilenet'] = models.mobilenet_v2(weights="DEFAULT", progress=True)
# DNNs['resnext50_32x4d'] = models.resnext50_32x4d(weights="DEFAULT", progress=True)
# DNNs['wide_resnet50_2'] = models.wide_resnet50_2(weights="DEFAULT", progress=True)
# DNNs['mnasnet'] = models.mnasnet1_0(weights="DEFAULT", progress=True)

In [15]:
# 上の中から試したいモデルを選んでください。最後のモデルが有効になります。
net = DNNs['resnet18']
#net = DNNs['squeezenet']
#net = DNNs['googlenet']
#net = DNNs['shufflenet']
#net = DNNs['mobilenet']
#net = DNNs['vgg16']
#net = DNNs['alexnet']

In [16]:
a_parameters = {name:param for name, param in net.named_parameters()}
a_modules = {name:param for name, param in net.named_modules()}

In [ ]:
#device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [17]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.Resize((224,224)),
    transforms.CenterCrop(224),
    transforms.ToTensor()])

# RGB 各チャンネルの平均と分散の定義。CNN 唯一の前処理
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

normalize = transforms.Normalize(mean=mean, std=std)
#mean = torch.mean(torch.tensor(means))
#std = torch.mean(torch.tensor(stds))

In [ ]:
# 乱数のシードを設定
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [18]:
# 入力画像の前処理をするクラス
# 訓練時と推論時で処理が異なる
class ImageTransform():
    """
    画像の前処理クラス。訓練時、検証時で異なる動作をする。
    画像のサイズをリサイズし、色を標準化する。
    訓練時は RandomResizedCrop と RandomHorizontalFlip で データ拡張

    Attributes
    ----------
    resize : int
        リサイズ先の画像の大きさ。
    mean : (R, G, B)
        各色チャネルの平均値。
    std : (R, G, B)
        各色チャネルの標準偏差。
    """
    def __init__(self, resize, mean, std):
        self.data_transform = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(
                    resize,
                    scale=(0.95, 1.0)),              # データ拡張
                    #scale=(0.8, 1.0)),              # データ拡張
                transforms.RandomHorizontalFlip(),  # データ拡張
                transforms.RandomAffine(
                    degrees=(-5,5),
                    #degrees=(-20,20),
                    translate=None,
                    #scale=[0.9,1.1]
                ),
                transforms.ToTensor(),          # テンソルに変換
                transforms.Normalize(mean, std) # 標準化
            ]),
            'val': transforms.Compose([
                transforms.Resize(resize),      # リサイズ
                transforms.CenterCrop(resize),  # 画像中央を resize×resize で切り取り
                transforms.ToTensor(),          # テンソルに変換
                transforms.Normalize(mean, std) # 標準化
            ])
        }

    def __call__(self, img, phase='train'):
        """
        Parameters
        ----------
        phase : 'train' or 'val'
            前処理のモードを指定。
        """
        return self.data_transform[phase](img)

In [ ]:
#help(transforms.RandomAffine)

## 3. PyTorch 用の DataSet の定義

In [ ]:
# 画像の前処理と処理済み画像の表示
size = 224
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

# Dataset の作成
class tlpa_sala_torch_Dataset(data.Dataset):
    """
    TLPA 画像のDatasetクラス。PyTorchのDatasetクラスを継承。

    Attributes
    ----------
    file_list : リスト
        画像のパスを格納したリスト
    transform : object
        前処理クラスのインスタンス
    phase : 'train' or 'test'
        学習か訓練かを設定する。
    """

    def __init__(self, file_list, name_dict, transform=None, phase='train'):
        self.file_list = file_list  # ファイルパスのリスト
        self.transform = transform  # 前処理クラスのインスタンス
        self.phase = phase  # train or valの指定
        self.namedict = name_dict

    def __len__(self):
        '''画像の枚数を返す'''
        return len(self.file_list)

    def __getitem__(self, index):
        '''
        前処理をした画像のTensor形式のデータとラベルを取得
        '''

        # index番目の画像をロード
        img_path = self.file_list[index]
        img = PILImage.open(img_path)  # [高さ][幅][色RGB]

        # 画像の前処理を実施
        img_transformed = self.transform(
            img, self.phase)  # torch.Size([3, 224, 224])

        # 画像のラベルをファイル名から抜き出す
        label = self.namedict[index]
        return img_transformed, label


train_dataset = tlpa_sala_torch_Dataset(file_list= _img_file_list,
                                        name_dict = _num2id,
                                        transform = ImageTransform(size, mean, std),
                                        phase='train')

val_dataset = tlpa_sala_torch_Dataset(file_list= _img_file_list,
                                      name_dict = _num2id,
                                      transform = ImageTransform(size, mean, std),
                                      phase='val')

# 動作確認
index = 3
print(train_dataset.__getitem__(index)[0].size())
print(train_dataset.__getitem__(index)[1])
print(train_dataset.__len__())

In [ ]:
# ミニバッチのサイズを指定
batch_size = 64

# DataLoaderを作成
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True)

val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False)

# 辞書型変数にまとめる
dataloaders_dict = {
    "train": train_dataloader,
    "val": val_dataloader}

# 動作確認
batch_iterator = iter(dataloaders_dict["train"])  # イテレータに変換
inputs, labels = next(
    batch_iterator)  # 1番目の要素を取り出す
print(inputs.size())
print(labels)

In [ ]:
# モデルのインスタンスを生成し，事前学習済の結合係数をロード
#use_pretrained = True  # 学習済みのパラメータを使用
#net = models.resnet18(pretrained=use_pretrained)
net.fc

## 6. 転移学習のための最終層の付け替え

In [22]:
# モデルの最終直下層の出力ユニット数を TLPA+SALA に合わせて 366 にする
net.fc = nn.Linear(in_features=512, out_features=_out_features)

# 訓練モードに設定
net.train();

## 4.  損失関数の定義

In [23]:
# 損失関数の設定
criterion = nn.CrossEntropyLoss()

In [ ]:
# 転移学習で学習させるパラメータを、変数params_to_updateに格納する
params_to_update = []

# 学習させるパラメータ名
update_param_names = ["fc.weight", "fc.bias"]

# 学習させるパラメータ以外は勾配計算をなくし、変化しないように設定
for name, param in net.named_parameters():
    if name in update_param_names:
        param.requires_grad = True
        params_to_update.append(param)
        print(name)
    else:
        param.requires_grad = False

# params_to_updateの中身を確認
print("-----------")
print(params_to_update)

## 5. 最適化関数の定義

In [25]:
# 最適化手法の設定
#optimizer = optim.SGD(params=params_to_update, lr=0.001, momentum=0.9)
#help(optim.Adam)
#optimizer = optim.SGD(params=params_to_update, lr=0.001, momentum=0.9)
optimizer = optim.Adam(params=params_to_update)

In [26]:
# モデルを学習させる関数
def train_model(net, dataloaders_dict, criterion, optimizer, num_epochs):

    # epochのループ
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch+1, num_epochs))
        print('-------------')

        # epochごとの学習と検証のループ
        for phase in ['train', 'val']:
            if phase == 'train':
                net.train()  # モデルを訓練モード
            else:
                net.eval()   # モデルを検証モード

            epoch_loss = 0.0  # epochの損失和
            epoch_corrects = 0  # epochの正解数

            # 未学習時の検証性能を確かめるため、epoch=0の訓練は省略
            if (epoch == 0) and (phase == 'train'):
                continue

            # データローダーからミニバッチを取り出すループ
            #for inputs, labels in tqdm(dataloaders_dict[phase]):
            # tqdm は要らん。冗長な出力になるだけ
            for inputs, labels in dataloaders_dict[phase]:
                # optimizerを初期化
                optimizer.zero_grad()

                # 順伝搬（forward）計算
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)  # 損失を計算
                    _, preds = torch.max(outputs, 1)  # ラベルを予測

                    # 訓練時はバックプロパゲーション
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                    # イタレーション結果の計算
                    # lossの合計を更新
                    epoch_loss += loss.item() * inputs.size(0)
                    # 正解数の合計を更新
                    epoch_corrects += torch.sum(preds == labels.data)

            # epochごとのlossと正解率を表示
            epoch_loss = epoch_loss / len(dataloaders_dict[phase].dataset)
            epoch_acc = epoch_corrects.double(
            ) / len(dataloaders_dict[phase].dataset)

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(
                phase, epoch_loss, epoch_acc))

In [ ]:
# for inputs, labels in dataloaders_dict['train']:
#     print(inputs.size(), labels)
#     output = net(inputs)
#     loss = criterion(output, labels)

In [ ]:
# # 保存してある訓練済モデルを読み込む
# import os
# HOME = os.environ['HOME']
# base = os.path.join(HOME, 'study/2020ccap/notebooks')
# saved_weight_file = os.path.join(base,'2020-0722tlpa_sala_resnet18_weights.pth')
# load_weights = torch.load(saved_weight_file)
# net.load_state_dict(load_weights)
# net.eval()

In [28]:
%%time
# 学習・検証を実行する
num_epochs=10
train_model(net, dataloaders_dict, criterion, optimizer, num_epochs=num_epochs)

Epoch 1/10
-------------
val Loss: 5.9401 Acc: 0.0055
Epoch 2/10
-------------
train Loss: 6.4821 Acc: 0.0055
val Loss: 5.8214 Acc: 0.0164
Epoch 3/10
-------------
train Loss: 5.7100 Acc: 0.0191
val Loss: 5.4456 Acc: 0.0301
Epoch 4/10
-------------
train Loss: 5.2930 Acc: 0.0383
val Loss: 5.1757 Acc: 0.0464
Epoch 5/10
-------------
train Loss: 4.9369 Acc: 0.0683
val Loss: 4.8952 Acc: 0.1311
Epoch 6/10
-------------
train Loss: 4.5987 Acc: 0.3033
val Loss: 4.5995 Acc: 0.2705
Epoch 7/10
-------------
train Loss: 4.2247 Acc: 0.5546
val Loss: 4.3077 Acc: 0.3880
Epoch 8/10
-------------
train Loss: 3.9022 Acc: 0.6858
val Loss: 3.9982 Acc: 0.5191
Epoch 9/10
-------------
train Loss: 3.6143 Acc: 0.7869
val Loss: 3.6637 Acc: 0.6475
Epoch 10/10
-------------
train Loss: 3.3218 Acc: 0.8689
val Loss: 3.3192 Acc: 0.7377
CPU times: user 11min 35s, sys: 1min 51s, total: 13min 27s
Wall time: 13min 43s


In [ ]:
# #saved_weight_file = '2020-0722tlpa_sala_resnet18_weights-2.pth'
# saved_weight_file = '2023_0718tlpa_sala_resnet18_weights.pth'
# torch.save(net.state_dict(), saved_weight_file)
# load_weights = torch.load(saved_weight_file)
# net.load_state_dict(load_weights)

In [ ]:
# #saved_weight_file = '2020-0722tlpa_sala_resnet18_weights-2.pth'
# #torch.save(net.state_dict(), saved_weight_file)
# load_weights = torch.load(saved_weight_file)
# net.load_state_dict(load_weights)

In [ ]:
#float_formatter = "{:.3f}".format
#np.set_printoptions(formatter={'float_kind':float_formatter})
# see https://note.nkmk.me/python-numpy-set-printoptions-float-formatter/
np.set_printoptions(formatter={'int': '{:3d}'.format, 'float_kind':'{:.3f}'.format})

def diagnose(no,
             name_dict=_num2id,
             num2word=_num2word,
             id2word=_id2word,
             filelist=_img_file_list,
             display=False,
             n_best=5):
    _id = name_dict[no]
    img_file = filelist[no]
    img = PILImage.open(img_file)

    label = num2word[no]

    # 画像の前処理と処理済み画像の表示
    size = 224
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    transform = ImageTransform(size, mean, std)
    img_transformed = transform(img, phase="val")  # torch.Size([3, 224, 224])

    # (色、高さ、幅)を (高さ、幅、色)に変換し、0-1に値を制限して表示
    if display:
        img_transformed_ = img_transformed.numpy().transpose((1, 2, 0))
        img_transformed_ = np.clip(img_transformed_, 0, 1)
        plt.axis(False); plt.imshow(img_transformed_);plt.show()

    # 認識の実施
    inputs = transform(img, phase='val')
    inputs_ = inputs.unsqueeze_(0)
    out = net(inputs_)
    outnp = out.detach().numpy()
    ids = np.argsort( - outnp[0])
    sftmx = softmax(-outnp[0])

    print(f'{no:3d}', end=" ")
    OK = True
    if _id == ids[0]:
        print('Hit ', end="")
    else:
        print(colored('Miss', 'red'), end="")
        OK = False

    print(ids[:n_best], end=" ")
    for id_ in ids[:n_best]:
        print(id2word[id_], end=" ")
    print(- np.sort(-sftmx)[:n_best])
    if not OK:
        img_transformed_ = img_transformed.numpy().transpose((1, 2, 0))
        img_transformed_ = np.clip(img_transformed_, 0, 1)
        #plt.title('正解:{0}, 出力:{1},{2},{3}'.format(vocabs_i2w[id],
        #                                          vocabs_i2w[ids[0]],
        #                                          vocabs_i2w[ids[1]],
        #                                          vocabs_i2w[ids[2]]))
        plt.figure(figsize=(3,3))
        plt.title((f'正解:{num2word[no]}',
                  f'出力:{id2word[ids[0]]},', #,
                  f'{id2word[ids[1]]},', #) #,
                  f'{id2word[ids[2]]}'))
        # plt.title('正解:{0}, 出力:{1},{2},{3}'.format(num2word[no],
        #                                           id2word[ids[0]],
        #                                           id2word[ids[1]],
        #                                           id2word[ids[2]]))
        plt.axis(False);plt.imshow(img_transformed_);plt.show()


for i in range(len(_img_file_list)): # tlpa_sala)):
    diagnose(i, display=False, n_best=5)

In [ ]:
import ccap
#importlib.reload(ccap)
from ccap import imagenetDataset

In [ ]:
imagenet = imagenetDataset()
for _ in range(3):
    num = np.random.choice(len(imagenet))
    imagenet.sample_and_show(num)

In [ ]:
# 出力結果からラベルを予測する後処理クラス
class ImageNetPredictor():
    """
    ImageNet データに対するモデルの出力からラベル出力

    Attributes:
        class_index : dictionary
        クラス index とラベル名 を対応させた辞書型変数
    """

    def __init__(self, class_index):
        self.class_index = class_index

    def predict_max(self, out):
        """
        最大値を与える ImageNet ラベル名を返す

        Parameters:
            out : torch.Size([1, 1000])  Net からの出力

        Returns:
            predicted_labels: [str]
            最も予測確率が高いラベルの名前
        """
        outnp = out.detach().numpy()
        ids = np.argsort(- outnp)
        predicted_labels = [self.class_index[id] for id in ids[0]]

        return ids[0], predicted_labels, softmax(outnp)

In [ ]:
# 下の no は 0 から 999 まで 1000 の ImageNet class に対応
no = np.random.choice(len(imagenet))
#print(f'no:{no}, ラベル:{imagenet.data[no]["label"]}')
img_file = imagenet.sample_image(no)
img = PILImage.open(img_file)
plt.title(f'{no}:{imagenet.data[no]["label"]}')
plt.axis(False); plt.imshow(img); plt.show()

In [ ]:
# 認識の実施
inputs = transform(img).unsqueeze_(0)  # torch.Size([1, 3, 224, 224])
out = net(inputs)  # torch.Size([1, 1000])

outnp = out.detach().numpy()
ids = np.argsort( - outnp)

n_best = 3
print(ids[0][:n_best])
for no in ids[0][:n_best]:
    print(imagenet(int(no))[1], end=" ")
    print(imagenet.getitem_from_wnid(imagenet(int(no))[1])['label'], end=" ")
    print(imagenet.getitem_from_wnid(imagenet(int(no))[1])['label_ja'], end=" ")
    print(imagenet.getitem_from_wnid(imagenet(int(no))[1])['definition'])
